In [ ]:
from google.colab import drive

drive.mount("/content/drive")


# Backfill historico das bases `falando_nela` no Colab

Este notebook orquestra uma coleta longa em modo producao, salvando tudo no Google Drive e usando `--resume` com `run_id`s fixos para permitir retomada apos queda de sessao.

A janela padrao tenta maximizar cobertura historica. Para analise, trate registros anteriores a 2010 como historicos e de menor confianca ate que a normalizacao/schema marquem isso explicitamente.

Regra de granularidade do corpus textual: arquivos em `raw/{source}/{dataset}/ano=YYYY/mes=MM/` devem conter apenas respostas de janelas mensais daquele mesmo mes. Consultas anuais ou trimestrais, quando suportadas pela fonte, podem existir como preflight de descoberta, mas precisam ficar em `metadata/` e nunca no corpus textual. Isso vale especialmente para `camara/plenario_discursos`, porque uma resposta maior que um mes pode misturar meses diferentes.

Estrategia recomendada para coletores textuais historicos quando o preflight for suportado pela fonte:

1. consultar o ano;
2. se o ano tiver retorno, for inconclusivo ou falhar, consultar trimestres;
3. se o trimestre tiver retorno, for inconclusivo ou falhar, consultar meses;
4. gravar texto analitico somente a partir das requisicoes mensais.

Excecao operacional: `senado/plenario_discursos` e `senado/congresso_discursos` usam o endpoint mensal `plenario/lista/discursos`, que rejeita janelas trimestrais/anuais; nesses casos, o caderno reduz vazios pelo inicio operacional de cada dataset.


In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)

for name in ["raw", "checkpoints", "logs", "manifests", "processed"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

print("FALANDO_NELA_DATA_ROOT=", os.environ["FALANDO_NELA_DATA_ROOT"])


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
print("Repositorio pronto em", REPO_DIR)
subprocess.run(["git", "status", "--short"], check=False)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


## Parametros

Por seguranca, a validacao curta fica ligada e o backfill completo fica desligado. Depois de conferir escrita no Drive, manifests e auditoria do layout raw, mude `RODAR_BACKFILL_COMPLETO` para `True`.

`DATA_INICIO_HISTORICO = "1900-01-01"` continua sendo o inicio amplo para bases que aceitam essa janela. Ha excecoes operacionais por fonte: `camara/plenario_discursos` comeca em `1946-01-01`; `senado/plenario_discursos` comeca em `1995-02-01`; `senado/congresso_discursos` comeca em `1996-05-01`. No endpoint de lista de discursos do Senado, consultas acima de um mes retornam erro, entao a reducao de vazios vem do recorte inicial, nao de preflight anual/trimestral. Periodos anteriores devem ser diagnosticos separados, se necessario. Consultas de preflight, quando existirem, ficam em `metadata/`; somente requisicoes mensais entram no corpus textual.

`RODAR_PARLAMENTARES_ANTES_TEXTOS=True` prepara `parlamentares/v1` antes dos coletores textuais completos, permitindo que `camara/plenario_discursos` e `camara/plenario_apartes` usem mandatos oficiais para reduzir consultas vazias.


In [ ]:
from datetime import datetime, timezone

DATA_INICIO_HISTORICO = "1900-01-01"
DATA_INICIO_CAMARA_PLENARIO_OFICIAL = "1946-01-01"
DATA_INICIO_SENADO_PLENARIO_OPERACIONAL = "1995-02-01"
DATA_INICIO_SENADO_CONGRESSO_OPERACIONAL = "1996-05-01"
DATA_FIM = "2026-05-28"
DATA_INICIO_ANALISE_RECOMENDADA = "2010-01-01"

RODAR_VALIDACAO_CURTA = True
RODAR_BACKFILL_COMPLETO = False
RODAR_PROCESSAMENTO = False
RODAR_PARLAMENTARES_ANTES_TEXTOS = True
RODAR_SAMPLES = False

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

COLETORES_TEXTUAIS = [
    {
        "nome": "senado-plenario",
        "module": "coleta.senado.plenario_discursos.collect",
        "run_id": "prod-historico-senado-plenario",
        "data_inicio": DATA_INICIO_SENADO_PLENARIO_OPERACIONAL,
        "validacao_inicio": "1995-03-01",
        "validacao_fim": "1995-03-31",
        "validacao_sample_limit": 10,
        "enabled": True,
    },
    {
        "nome": "senado-congresso",
        "module": "coleta.senado.congresso_discursos.collect",
        "run_id": "prod-historico-senado-congresso",
        "data_inicio": DATA_INICIO_SENADO_CONGRESSO_OPERACIONAL,
        "validacao_inicio": "2000-03-01",
        "validacao_fim": "2000-03-31",
        "validacao_sample_limit": None,
        "enabled": True,
    },
    {
        "nome": "senado-ccj",
        "module": "coleta.senado.ccj_notas.collect",
        "run_id": "prod-historico-senado-ccj",
        "validacao_inicio": "2012-03-01",
        "validacao_fim": "2012-03-31",
        "validacao_sample_limit": 5,
        "enabled": True,
    },
    {
        "nome": "senado-pareceres-pec",
        "module": "coleta.senado.pareceres_pec.collect",
        "run_id": "prod-historico-senado-pareceres-pec",
        "validacao_inicio": "2019-01-01",
        "validacao_fim": "2019-12-31",
        "validacao_sample_limit": 3,
        "enabled": True,
    },
    {
        "nome": "camara-plenario",
        "module": "coleta.camara.plenario_discursos.collect",
        "run_id": "prod-historico-camara-plenario",
        "data_inicio": DATA_INICIO_CAMARA_PLENARIO_OFICIAL,
        "validacao_inicio": "1995-03-01",
        "validacao_fim": "1995-03-31",
        "validacao_sample_limit": 3,
        "enabled": True,
    },
    {
        "nome": "camara-ccjc",
        "module": "coleta.camara.ccjc_eventos.collect",
        "run_id": "prod-historico-camara-ccjc",
        "validacao_inicio": "2019-01-01",
        "validacao_fim": "2019-01-31",
        "validacao_sample_limit": 3,
        "enabled": True,
    },
    {
        "nome": "camara-pareceres-pec",
        "module": "coleta.camara.pareceres_pec.collect",
        "run_id": "prod-historico-camara-pareceres-pec",
        "validacao_inicio": "2019-01-01",
        "validacao_fim": "2019-12-31",
        "validacao_sample_limit": 3,
        "enabled": True,
    },
]

PARLAMENTARES_PROCESSADOS_RUN_ID = "processed-historico-parlamentares-v1"

PARLAMENTARES = {
    "nome": "parlamentares",
    "module": "coleta.parlamentares.collect",
    "run_id": "prod-historico-parlamentares",
    "validacao_inicio": "1995-01-01",
    "validacao_fim": "1995-12-31",
    "validacao_sample_limit": 10,
    "extra_args": ["--skip-existing-id-scan", "--skip-detail-endpoints"],
    "enabled": True,
}

print("RUN_STAMP=", RUN_STAMP)
print("Janela historica ampla:", DATA_INICIO_HISTORICO, "a", DATA_FIM)
print("Camara Plenario oficial:", DATA_INICIO_CAMARA_PLENARIO_OFICIAL, "a", DATA_FIM)
print("Senado Plenario operacional:", DATA_INICIO_SENADO_PLENARIO_OPERACIONAL, "a", DATA_FIM)
print("Senado Congresso operacional:", DATA_INICIO_SENADO_CONGRESSO_OPERACIONAL, "a", DATA_FIM)
print("Analise recomendada por default:", DATA_INICIO_ANALISE_RECOMENDADA, "em diante")
print("Preparo de parlamentares antes dos textos:", RODAR_PARLAMENTARES_ANTES_TEXTOS)


In [ ]:
import json
import subprocess
import sys
from datetime import date
from pathlib import Path

collection_results = []


def run_streamed(cmd, *, label, cwd=REPO_DIR):
    print("\n===", label, "===")
    print("Rodando:", " ".join(str(part) for part in cmd))
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        lines.append(line.rstrip("\n"))
    returncode = process.wait()
    manifest_candidates = [
        line.strip()
        for line in lines
        if line.strip().endswith(".json") and "/manifests/" in line.strip()
    ]
    result = {
        "label": label,
        "cmd": [str(part) for part in cmd],
        "returncode": returncode,
        "manifest_path": manifest_candidates[-1] if manifest_candidates else None,
    }
    collection_results.append(result)
    print("\nRETURN CODE:", returncode)
    if result["manifest_path"]:
        print("MANIFEST:", result["manifest_path"])
    return result


def collector_cmd(config, data_inicio, data_fim, *, run_id, sample=False, sample_limit=None, resume=True):
    cmd = [
        sys.executable,
        "-m",
        config["module"],
        "--mode",
        "prod",
        "--run-id",
        run_id,
        "--data-inicio",
        data_inicio,
        "--data-fim",
        data_fim,
    ]
    if resume:
        cmd.append("--resume")
    if sample:
        cmd.append("--sample")
    else:
        cmd.append("--no-sample")
    if sample_limit is not None:
        cmd.extend(["--sample-limit", str(sample_limit)])
    cmd.extend(config.get("extra_args", []))
    return cmd


def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def manifest_path_for(run_id):
    return DATA_ROOT / "manifests" / f"{run_id}.json"


def autosave_path_for(run_id):
    return DATA_ROOT / "manifests" / f"{run_id}.autosave.json"


def log_path_for(run_id):
    return DATA_ROOT / "logs" / f"{run_id}.jsonl"


def manifest_for(run_id):
    return read_json(manifest_path_for(run_id)) or read_json(autosave_path_for(run_id))


def raw_root_for_manifest(manifest):
    source = manifest.get("source")
    dataset = manifest.get("dataset")
    if not source or not dataset:
        return None
    return DATA_ROOT / "raw" / source / dataset


def parse_iso_date(value):
    return date.fromisoformat(str(value))


def month_from_corpus_path(path):
    path = Path(path)
    mes_part = path.parent.name
    ano_part = path.parent.parent.name
    if not (ano_part.startswith("ano=") and mes_part.startswith("mes=")):
        return None
    return int(ano_part.removeprefix("ano=")), int(mes_part.removeprefix("mes="))


def iter_jsonl(path, *, max_records=None):
    emitted = 0
    invalid_lines = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                record = json.loads(stripped)
            except json.JSONDecodeError as exc:
                invalid_lines.append((line_number, str(exc), stripped[:160]))
                continue
            yield record
            emitted += 1
            if max_records is not None and emitted >= max_records:
                break
    if invalid_lines:
        print("  JSONL invalido ignorado em", Path(path), "linhas=", len(invalid_lines))
        for line_number, error, prefix in invalid_lines[:3]:
            print("   ", {"linha": line_number, "erro": error, "prefixo": prefix})


def record_period(record):
    periodo = record.get("periodo") or {}
    inicio = periodo.get("data_inicio")
    fim = periodo.get("data_fim")
    if not inicio or not fim:
        return None
    return parse_iso_date(inicio), parse_iso_date(fim)


def audit_raw_layout_for_run(run_id, *, max_files=200, max_records_per_file=200, strict=True):
    manifest = manifest_for(run_id)
    print("\nAUDITORIA RAW:", run_id)
    if not manifest:
        print("  sem manifest/autosave; auditoria pulada")
        return []

    raw_root = raw_root_for_manifest(manifest)
    if raw_root is None:
        print("  manifest sem source/dataset; auditoria pulada")
        return []
    if not raw_root.exists():
        print("  raw ainda nao existe:", raw_root)
        return []

    metadata_file = raw_root / "metadata" / f"{run_id}.jsonl"
    monthly_files = sorted(raw_root.glob(f"ano=*/mes=*/{run_id}.jsonl"))
    print("  raw_root:", raw_root)
    print("  metadata:", metadata_file.exists(), metadata_file)
    print("  arquivos de corpus mensal:", len(monthly_files))

    if metadata_file.exists():
        metadata_counts = {}
        for record in iter_jsonl(metadata_file, max_records=max_records_per_file):
            record_type = record.get("record_type")
            metadata_counts[record_type] = metadata_counts.get(record_type, 0) + 1
        print("  record_types em metadata amostrados:", metadata_counts)

    issues = []
    sampled_files = monthly_files[:max_files]
    for jsonl_path in sampled_files:
        path_month = month_from_corpus_path(jsonl_path)
        if path_month is None:
            issues.append((str(jsonl_path), "caminho de corpus invalido", None))
            continue
        path_year, path_month_number = path_month
        for record in iter_jsonl(jsonl_path, max_records=max_records_per_file):
            period = record_period(record)
            if period is None:
                continue
            start, end = period
            same_month = start.year == end.year and start.month == end.month
            matches_path = start.year == path_year and start.month == path_month_number
            if not same_month or not matches_path:
                issues.append((str(jsonl_path), record.get("source_id"), record.get("periodo")))
                break

    if len(monthly_files) > len(sampled_files):
        print(f"  auditoria amostrou {len(sampled_files)} de {len(monthly_files)} arquivos mensais")
    if issues:
        print("  problemas encontrados:")
        for issue in issues[:20]:
            print("   ", issue)
        if strict:
            raise AssertionError("Corpus mensal contem registro com periodo fora do mes do caminho")
    else:
        print("  OK: corpus amostrado contem apenas janelas mensais compatíveis com o caminho")
    return issues


## Validacao curta

Esta etapa roda em `--mode prod` para testar escrita no Drive, mas usa `--sample` e janelas pequenas. Ela nao substitui a coleta completa.

Depois de cada coletor, o notebook audita o layout raw. Se houver arquivo em `ano=YYYY/mes=MM/`, seus registros precisam ter `periodo` dentro do mesmo mes do caminho. Respostas de descoberta maiores que um mes devem aparecer somente em `metadata/`.


In [ ]:
if RODAR_VALIDACAO_CURTA:
    for config in COLETORES_TEXTUAIS + [PARLAMENTARES]:
        if not config.get("enabled", True):
            continue
        run_id = f"validacao-{config['nome']}-{RUN_STAMP}"
        cmd = collector_cmd(
            config,
            config["validacao_inicio"],
            config["validacao_fim"],
            run_id=run_id,
            sample=True,
            sample_limit=config.get("validacao_sample_limit"),
            resume=True,
        )
        run_streamed(cmd, label=f"validacao {config['nome']}")
        audit_raw_layout_for_run(run_id, strict=True)
else:
    print("Validacao curta desligada.")


## Backfill completo

Esta celula pode levar muitas horas ou dias. Os `run_id`s sao fixos para cada dataset; se o Colab cair, rode a mesma celula novamente com `--resume`.

A auditoria de layout roda apos cada coletor. Em runs muito grandes, ela amostra arquivos JSONL para evitar uma varredura pesada; use a celula de inspecao para repetir a auditoria quando necessario.

Quando `RODAR_PARLAMENTARES_ANTES_TEXTOS=True`, a celula roda primeiro `coleta.parlamentares.collect` e `processamento.parlamentares`. Isso gera `parlamentares_periodos` para que os coletores lentos da Camara consultem apenas deputados com mandato vigente em cada ano.

Para a etapa preparatoria de parlamentares, o caderno passa `--skip-existing-id-scan`: a lista oficial por periodo basta para montar `parlamentares_periodos`, e isso evita uma varredura pesada de todo o Drive antes do primeiro progresso.

A mesma etapa tambem passa `--skip-detail-endpoints` para montar rapidamente o plano de mandatos a partir das listas oficiais por legislatura. Rode a coleta completa de parlamentares depois quando precisar enriquecer genero/detalhe historico para analise.


In [ ]:
if RODAR_BACKFILL_COMPLETO:
    if PARLAMENTARES.get("enabled", True) and RODAR_PARLAMENTARES_ANTES_TEXTOS:
        cmd = collector_cmd(
            PARLAMENTARES,
            DATA_INICIO_HISTORICO,
            DATA_FIM,
            run_id=PARLAMENTARES["run_id"],
            sample=False,
            sample_limit=None,
            resume=True,
        )
        run_streamed(cmd, label="backfill parlamentares")
        audit_raw_layout_for_run(PARLAMENTARES["run_id"], max_files=200, max_records_per_file=200, strict=True)
        run_streamed(
            [
                sys.executable,
                "-m",
                "processamento.parlamentares",
                "--mode",
                "prod",
                "--data-root",
                str(DATA_ROOT),
                "--run-id",
                PARLAMENTARES_PROCESSADOS_RUN_ID,
                "--data-inicio",
                DATA_INICIO_HISTORICO,
                "--data-fim",
                DATA_FIM,
                "--overwrite",
            ],
            label="processamento parlamentares/v1 para plano de mandatos",
        )

    for config in COLETORES_TEXTUAIS:
        if not config.get("enabled", True):
            continue
        data_inicio = config.get("data_inicio", DATA_INICIO_HISTORICO)
        cmd = collector_cmd(
            config,
            data_inicio,
            DATA_FIM,
            run_id=config["run_id"],
            sample=False,
            sample_limit=None,
            resume=True,
        )
        print("Janela do coletor:", data_inicio, "a", DATA_FIM)
        run_streamed(cmd, label=f"backfill {config['nome']}")
        audit_raw_layout_for_run(config["run_id"], max_files=200, max_records_per_file=200, strict=True)

    if PARLAMENTARES.get("enabled", True) and not RODAR_PARLAMENTARES_ANTES_TEXTOS:
        cmd = collector_cmd(
            PARLAMENTARES,
            DATA_INICIO_HISTORICO,
            DATA_FIM,
            run_id=PARLAMENTARES["run_id"],
            sample=False,
            sample_limit=None,
            resume=True,
        )
        run_streamed(cmd, label="backfill parlamentares")
        audit_raw_layout_for_run(PARLAMENTARES["run_id"], max_files=200, max_records_per_file=200, strict=True)
else:
    print("Backfill completo desligado. Mude RODAR_BACKFILL_COMPLETO para True nos parametros.")


## Inspecao de manifests, logs e layout raw

Use esta celula depois de validacao, depois de cada retomada, e antes do processamento. A auditoria reforca que corpus textual continua mensal mesmo quando algum coletor usa preflight anual/trimestral em `metadata/`.


In [ ]:
def print_run_summary(run_id):
    manifest = manifest_for(run_id)
    print("\nRUN:", run_id)
    if not manifest:
        print("  sem manifest/autosave")
        return
    keys = [
        "source",
        "dataset",
        "mode",
        "sample",
        "data_inicio",
        "data_fim",
        "status",
        "errors",
        "record_counts",
        "partition_counts",
    ]
    print(json.dumps({key: manifest.get(key) for key in keys}, ensure_ascii=False, indent=2, default=str))
    checkpoint_path = manifest.get("checkpoint_path")
    if checkpoint_path:
        checkpoint = read_json(checkpoint_path)
        if checkpoint:
            failed = checkpoint.get("failed_partitions", {})
            current_run = (checkpoint.get("runs", {}) or {}).get(run_id, {})
            run_failed = current_run.get("failed_partitions", {}) if isinstance(current_run, dict) else {}
            if failed or run_failed:
                print("  failed_partitions:", sorted(set(failed) | set(run_failed))[:30])
    log_path = log_path_for(run_id)
    if log_path.exists():
        tail = log_path.read_text(encoding="utf-8").splitlines()[-5:]
        print("  ultimas linhas do log:")
        for line in tail:
            print("   ", line[:500])


run_ids = [config["run_id"] for config in COLETORES_TEXTUAIS if config.get("enabled", True)]
if PARLAMENTARES.get("enabled", True):
    run_ids.append(PARLAMENTARES["run_id"])

for run_id in run_ids:
    print_run_summary(run_id)
    audit_raw_layout_for_run(run_id, max_files=200, max_records_per_file=200, strict=True)

if collection_results:
    print("\nResultados desta sessao:")
    print(json.dumps(collection_results, ensure_ascii=False, indent=2))


## Normalizacao e Parquets

Rode somente depois de conferir que os manifests do backfill estao aceitaveis. `--overwrite` substitui uma saida processada com o mesmo `run_id`, nao apaga o raw.


In [ ]:
PROCESSED_RUN_ID = "processed-textos-v1-historico"
PARQUET_RUN_ID = "parquet-textos-v1-historico"

if RODAR_PROCESSAMENTO:
    run_streamed(
        [
            sys.executable,
            "-m",
            "processamento.normalizacao",
            "--mode",
            "prod",
            "--data-root",
            str(DATA_ROOT),
            "--run-id",
            PROCESSED_RUN_ID,
            "--overwrite",
        ],
        label="normalizacao textos_parlamentares/v1",
    )
    run_streamed(
        [
            sys.executable,
            "-m",
            "processamento.parquet",
            "--profile",
            "colab",
            "--data-root",
            str(DATA_ROOT),
            "--run-id",
            PARQUET_RUN_ID,
            "--overwrite",
        ],
        label="geracao de Parquets",
    )
else:
    print("Processamento desligado. Mude RODAR_PROCESSAMENTO para True nos parametros.")


## Samples para baixar/validar localmente

Esta etapa gera ZIPs pequenos a partir do processed completo no Drive. Use depois dos Parquets.


In [ ]:
SAMPLES_RUN_ID = "samples-textos-v1-historico"

if RODAR_SAMPLES:
    run_streamed(
        [
            sys.executable,
            "-m",
            "processamento.samples",
            "--profile",
            "colab",
            "--data-root",
            str(DATA_ROOT),
            "--run-id",
            SAMPLES_RUN_ID,
            "--include-parquet",
            "--overwrite",
        ],
        label="geracao de samples ZIP",
    )
else:
    print("Samples desligados. Mude RODAR_SAMPLES para True nos parametros.")


## Checagem rapida dos Parquets

Esta celula lista arquivos Parquet gerados e estima cobertura anual quando `pandas`/`pyarrow` estiverem disponiveis.


In [ ]:
PARQUET_ROOT = DATA_ROOT / "processed" / "textos_parlamentares" / "v1" / "parquet"
print("PARQUET_ROOT=", PARQUET_ROOT)

if not PARQUET_ROOT.exists():
    print("Parquets ainda nao encontrados.")
else:
    parquet_files = sorted(PARQUET_ROOT.glob("*.parquet"))
    print("Arquivos:", [path.name for path in parquet_files])
    try:
        import pandas as pd
        for path in parquet_files:
            df = pd.read_parquet(path, columns=["source", "dataset", "ano", "texto_id"])
            coverage = df.groupby("ano")["texto_id"].count().sort_index()
            print("\n", path.name)
            print("linhas=", len(df), "primeiro_ano=", coverage.index.min() if len(coverage) else None, "ultimo_ano=", coverage.index.max() if len(coverage) else None)
            print(coverage.tail(10).to_string())
    except Exception as exc:
        print("Nao foi possivel abrir Parquets para resumo:", repr(exc))
